# Lab: building and testing a polygenic risk score

In this lab you will build a polygenic risk score (PRS) for type 2 diabetes and measure how well it works,
following the three-sample design from *Calculating PRS: workflow and data*:

- a **base GWAS** gives the effect sizes, as summary statistics only;
- a **target cohort** with genotypes and type 2 diabetes status is split into a **training** half,
  used to choose the P-value threshold, and a **test** half, used to measure how well the final score works.

The lab has three parts:

1. **Python**: quality control of the summary statistics, a PRS by hand, and the winner's curse.
2. **R**: how well does the score work? Odds ratio per SD, AUC and risk by decile.
3. **PLINK**: clumping and thresholding at several P-value thresholds.

The data are simulated: 2,000 SNPs in 100 LD blocks on 10 chromosomes, 100 of them (5%) causal.
Type 2 diabetes status follows a liability-threshold model with a prevalence of 10%.

See *Running the lab notebooks* in the book for where to run this notebook.

## Setup

In [ ]:
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm, rankdata

for tool in ["plink", "R"]:
    print(f"{tool:6s}", "found" if shutil.which(tool) else "NOT found (that part of the lab will not run)")

## Simulate the study

Run this cell once; it takes about half a minute. You do not need to follow every line, but note what it writes:

| File | Contents |
|---|---|
| `base_gwas.txt` | base GWAS summary statistics (20,000 people): `SNP CHR BP A1 A2 BETA SE P`, where `A1` is the effect allele and `BETA` the log odds ratio |
| `replication_gwas.txt` | a second, independent GWAS of the same size (used in Exercise 3) |
| `target.ped`, `target.map` | genotypes and type 2 diabetes status (1 = control, 2 = case) for 4,000 target individuals |
| `train.txt`, `test.txt` | the IDs of the training and test halves |

Within an LD block, neighbouring SNPs are correlated, as in *Clumping and thresholding*.

In [ ]:
rng = np.random.default_rng(2027)
n_chrom, blocks_per_chrom, block_size = 10, 10, 20
n_blocks = n_chrom * blocks_per_chrom
M = n_blocks * block_size                    # 2,000 SNPs
K, h2, n_causal, rho = 0.10, 0.15, 100, 0.92  # prevalence, SNP heritability, causal SNPs, LD between neighbours

freq = rng.uniform(0.05, 0.5, M)             # frequency of the allele we count
pairs = np.array([("A", "C"), ("A", "G"), ("C", "A"), ("C", "T"), ("G", "A"), ("G", "T"), ("T", "C"), ("T", "G")])
alleles = pairs[rng.integers(0, len(pairs), M)]          # column 0: counted allele, column 1: other allele
snp = np.array([f"rs{100001 + j}" for j in range(M)])
chrom = np.repeat(np.arange(1, n_chrom + 1), blocks_per_chrom * block_size)
bp = np.array([1_000_000 * (1 + (j // block_size) % blocks_per_chrom) + 10_000 * (j % block_size) for j in range(M)])

def genotypes(n):
    # two haplotypes per person; within a block, SNPs follow a correlated (AR-1) latent variable
    G = np.zeros((n, M), np.int8)
    for _ in range(2):
        for b in range(n_blocks):
            x = rng.standard_normal(n)
            for k in range(block_size):
                j = b * block_size + k
                if k:
                    x = rho * x + np.sqrt(1 - rho**2) * rng.standard_normal(n)
                G[:, j] += x < norm.ppf(freq[j])
    return G

causal = np.sort(rng.choice(M, n_causal, replace=False))
beta_true = np.zeros(M)
beta_true[causal] = rng.standard_normal(n_causal)
g_ref = genotypes(10_000) @ beta_true
g_mean, g_sd = g_ref.mean(), g_ref.std()

def t2d_status(G):
    g = (G @ beta_true - g_mean) / g_sd * np.sqrt(h2)                  # genetic liability
    liability = g + rng.standard_normal(len(g)) * np.sqrt(1 - h2)      # plus environment
    return (liability > norm.ppf(1 - K)).astype(int)                   # top 10% = cases

def gwas(n):
    G = genotypes(n); y = t2d_status(G)
    a = G[y == 1].sum(0); b = 2 * y.sum() - a                          # case alleles: counted, other
    c = G[y == 0].sum(0); d = 2 * (1 - y).sum() - c                    # control alleles
    beta = np.log(a * d / (b * c)); se = np.sqrt(1/a + 1/b + 1/c + 1/d)
    return pd.DataFrame({"SNP": snp, "CHR": chrom, "BP": bp, "A1": alleles[:, 0], "A2": alleles[:, 1],
                         "BETA": beta.round(5), "SE": se.round(5), "P": 2 * norm.sf(np.abs(beta / se))})

base, replication = gwas(20_000), gwas(20_000)

# three problems for the quality-control step, all at genome-wide significant SNPs
sig = base.index[base.P < 5e-8].to_numpy()
i_swap, i_ambig, i_missing = sig[[2, 5, 8]]
base.loc[i_swap, ["A1", "A2"]] = base.loc[i_swap, ["A2", "A1"]].to_numpy()     # reported for the other allele
base.loc[i_swap, "BETA"] *= -1
alleles[i_ambig] = ("A", "T")                                                   # a strand-ambiguous SNP
base.loc[i_ambig, ["A1", "A2"]] = ["A", "T"]
replication.loc[i_ambig, ["A1", "A2"]] = ["A", "T"]

for df, name in [(base, "base_gwas.txt"), (replication, "replication_gwas.txt")]:
    df["P"] = df["P"].map(lambda p: f"{p:.3g}")
    df.to_csv(name, sep=" ", index=False)

# target cohort; one base SNP is not genotyped in the target data
n_target = 4000
G = genotypes(n_target); y = t2d_status(G)
keep = np.setdiff1d(np.arange(M), [i_missing])
a1, a2 = alleles[keep, 0], alleles[keep, 1]
geno = np.where(G[:, keep] == 2, np.char.add(np.char.add(a1, " "), a1),
       np.where(G[:, keep] == 1, np.char.add(np.char.add(a1, " "), a2), np.char.add(np.char.add(a2, " "), a2)))
sex = rng.integers(1, 3, n_target)                     # PLINK: 1 = male, 2 = female
with open("target.ped", "w") as fh:
    for i in range(n_target):
        fh.write(f"T{i} T{i} 0 0 {sex[i]} {y[i] + 1} " + " ".join(geno[i]) + "\n")
pd.DataFrame({"CHR": chrom[keep], "SNP": snp[keep], "CM": 0, "BP": bp[keep]}).to_csv(
    "target.map", sep=" ", header=False, index=False)

ids = np.array([f"T{i}" for i in rng.permutation(n_target)])
for half, name in [(ids[:n_target // 2], "train.txt"), (ids[n_target // 2:], "test.txt")]:
    pd.DataFrame({"FID": half, "IID": half}).to_csv(name, sep=" ", header=False, index=False)

print(f"{y.sum()} cases and {n_target - y.sum()} controls in the target cohort")

PLINK's binary format (`.bed/.bim/.fam`) is faster than text. We convert the target data and write the
genotypes as allele counts (`--recode A`) so Python and R can read them.

In [ ]:
%%bash
plink --file target --make-bed --out target > /dev/null
plink --bfile target --recode A --out target > /dev/null
head -3 target.bim
cut -d " " -f 1-9 target.raw | head -3

Each column of `target.raw` is named `SNP_allele`, for example `rs100001_G`: the number (0, 1 or 2) is the count
of **that** allele. PLINK counts its `A1` allele, the minor allele in the target data, which is not always the
base GWAS effect allele.

## Part 1 — Python: a PRS by hand

### Quality control: line up the alleles

A PRS adds up effect sizes for the effect allele, so for every SNP the base and target data must refer to the same
allele (*Quality control*). We compare the base `A1/A2` with the target `A1/A2` from `target.bim`.

In [ ]:
base = pd.read_csv("base_gwas.txt", sep=" ")
bim = pd.read_csv("target.bim", sep="\t", header=None, names=["CHR", "SNP", "CM", "BP", "T_A1", "T_A2"])
m = base.merge(bim[["SNP", "T_A1", "T_A2"]], on="SNP", how="left")

complement = {"A": "T", "T": "A", "C": "G", "G": "C"}
def classify(r):
    if pd.isna(r.T_A1):
        return "missing"                                   # not genotyped in the target data
    if complement[r.A1] == r.A2:
        return "ambiguous"                                 # A/T or C/G: strand cannot be told
    if (r.A1, r.A2) == (r.T_A1, r.T_A2):
        return "match"
    if (r.A1, r.A2) == (r.T_A2, r.T_A1):
        return "swap"                                      # same alleles, other one counted
    return "mismatch"

m["status"] = m.apply(classify, axis=1)
print(m.status.value_counts())
m[(m.status != "match") & (m.P < 5e-8)][["SNP", "A1", "A2", "T_A1", "T_A2", "BETA", "P", "status"]]

**Exercise 1.** For each of the three genome-wide significant SNPs shown above, explain what is wrong and what to
do: keep it as it is, keep it with $\beta$ multiplied by −1, or drop it. Why can we not simply flip the strand-ambiguous
SNP? (Look at the `swap` row: what is the base GWAS effect allele, and which allele does the target data count?)

The cell below applies these rules. It writes `base_qc.txt`, the cleaned summary statistics that we use from now on.
Most of the SNPs in the `swap` count above are not errors in the base GWAS: PLINK simply counts the target minor
allele, which is sometimes the other allele.

In [ ]:
qc = m[m.status.isin(["match", "swap"])].copy()
qc["BETA_T"] = np.where(qc.status == "swap", -qc.BETA, qc.BETA)       # effect of the allele the target counts
qc[["SNP", "CHR", "BP", "A1", "A2", "BETA", "SE", "P"]].to_csv("base_qc.txt", sep=" ", index=False)
print(len(qc), "SNPs pass QC")

### A PRS from the genome-wide significant SNPs

$$\mathrm{PRS}_j = \sum_i \beta_i\, G_{ij}$$

where $G_{ij}$ is the number of effect alleles person $j$ carries at SNP $i$. We use the SNPs with $P < 5\times10^{-8}$
and standardise the score (mean 0, SD 1) so that it is easy to interpret.

In [ ]:
raw = pd.read_csv("target.raw", sep=r"\s+")
geno = raw.iloc[:, 6:]
geno.columns = [c.rsplit("_", 1)[0] for c in geno.columns]              # rs100001_G -> rs100001
status = (raw.PHENOTYPE == 2).astype(int)                                # 1 = type 2 diabetes

gws = qc[qc.P < 5e-8]
prs = geno[gws.SNP].to_numpy() @ gws.BETA_T.to_numpy()
prs_z = (prs - prs.mean()) / prs.std()
print(len(gws), "SNPs in the score")

plt.hist(prs_z[status == 0], bins=40, density=True, alpha=0.6, label="controls")
plt.hist(prs_z[status == 1], bins=40, density=True, alpha=0.6, label="type 2 diabetes")
plt.xlabel("PRS (standardised)"); plt.ylabel("density"); plt.legend(); plt.show()
print(f"mean PRS: cases {prs_z[status == 1].mean():.2f}, controls {prs_z[status == 0].mean():.2f}")

**Exercise 2.** The two distributions overlap a lot. What does this tell you about using the PRS to predict who
will develop type 2 diabetes? Re-run the cell using `qc` instead of `gws` (all SNPs that passed QC, with no P-value
threshold). Does the separation between cases and controls look better or worse?

### The winner's curse

SNPs that pass a strict threshold were selected partly because chance pushed their estimates up
(*Shrinkage and the winner's curse*). An independent replication GWAS of the same size shows how much.

In [ ]:
rep = pd.read_csv("replication_gwas.txt", sep=" ")[["SNP", "A1", "BETA"]].rename(columns={"A1": "R_A1", "BETA": "R_BETA"})
w = gws.merge(rep, on="SNP")
w["R_BETA"] = np.where(w.R_A1 == w.A1, w.R_BETA, -w.R_BETA)               # same effect allele as the base GWAS
w["sign"] = np.sign(w.BETA)                                              # compare effect sizes on the risk scale
plt.scatter(w.BETA * w.sign, w.R_BETA * w.sign)
lim = [0, (w.BETA * w.sign).max() * 1.1]
plt.plot(lim, lim, "k--", lw=1)
plt.xlabel("|β| in the base GWAS (discovery)"); plt.ylabel("β in the replication GWAS")
plt.show()

**Exercise 3.** Most points fall below the dashed line, where the two estimates would be equal. By what percentage,
on average, does the base GWAS overestimate the effects of these SNPs? (Compute
`(w.BETA * w.sign).mean()` and `(w.R_BETA * w.sign).mean()`.) Now repeat this for the SNPs with $10^{-5} \le P < 10^{-3}$ in the base GWAS
(use `qc` instead of `gws`). Are their effects more or less inflated, and why?

## Part 2 — R: how well does the score work?

We send the standardised PRS and case status to R. A logistic regression of status on the standardised PRS gives
the **odds ratio per standard deviation** of the score, the usual way PRS effects are reported.

In [ ]:
%load_ext rpy2.ipython
prs_df = pd.DataFrame({"prs": prs_z, "t2d": status})

In [ ]:
%%R -i prs_df
fit <- glm(t2d ~ prs, family = binomial, data = prs_df)
print(summary(fit)$coefficients)
cat("\nOR per SD of PRS =", round(exp(coef(fit)["prs"]), 2), "\n")

# AUC: the probability that a random case scores higher than a random control
auc <- function(score, y) {
  r <- rank(score); n1 <- sum(y == 1); n0 <- sum(y == 0)
  (sum(r[y == 1]) - n1 * (n1 + 1) / 2) / (n1 * n0)
}
cat("AUC =", round(auc(prs_df$prs, prs_df$t2d), 3), "\n")

An AUC of 0.5 means the score is no better than a coin toss; 1 would be perfect. Next, the **risk gradient**:
the proportion with type 2 diabetes in each tenth (decile) of the PRS, as in *Risk stratification, not diagnosis*.

In [ ]:
%%R
prs_df$decile <- cut(prs_df$prs, quantile(prs_df$prs, 0:10 / 10), labels = 1:10, include.lowest = TRUE)
risk <- tapply(prs_df$t2d, prs_df$decile, mean)
print(round(100 * risk, 1))
barplot(100 * risk, xlab = "PRS decile", ylab = "type 2 diabetes (%)", col = "#4C72B0")
abline(h = 100 * mean(prs_df$t2d), lty = 2)

**Exercise 4.** Compare the **top 5%** of the PRS with everyone else.

- What is the absolute risk (the proportion with type 2 diabetes) in each group?
- What is the relative risk (top 5% ÷ everyone else)?
- Write one sentence for a patient in the top 5% that gives both numbers. Which number sounds more alarming,
  and which is more useful to the patient?

Hint: `top <- prs_df$prs >= quantile(prs_df$prs, 0.95)`.

In [ ]:
%%R
# Exercise 4

## Part 3 — PLINK: clumping and thresholding

The score in Parts 1 and 2 used only the genome-wide significant SNPs. Clumping and thresholding (C+T,
*Clumping and thresholding*) tries several P-value thresholds, keeping one index SNP per LD signal.

**Step 1: clump.** PLINK ranks the SNPs in `base_qc.txt` by P-value and, using LD (r²) estimated from the target
genotypes, removes SNPs with r² > 0.1 to a more significant index SNP within 250 kb.

In [ ]:
%%bash
plink --bfile target --clump base_qc.txt --clump-p1 1 --clump-p2 1 --clump-r2 0.1 --clump-kb 250 \
      --out target > /dev/null
head -4 target.clumped | cut -c 1-100

Each row of `target.clumped` is one clump: `SNP` is its index SNP and `SP2` lists the SNPs clumped with it.
We keep the summary statistics of the index SNPs only.

In [ ]:
index_snps = pd.read_csv("target.clumped", sep=r"\s+").SNP
clumped = qc[qc.SNP.isin(index_snps)]
clumped[["SNP", "CHR", "BP", "A1", "A2", "BETA", "SE", "P"]].to_csv("base_clumped.txt", sep=" ", index=False)
print(f"{len(clumped)} index SNPs from {len(qc)} SNPs")

**Step 2: score at several thresholds.** `--score` adds up `BETA` for each person's copies of the base `A1` allele;
PLINK matches alleles by name, so the swapped SNPs are handled automatically. `--q-score-range` makes one score per
P-value range in `ranges.txt`, taking each SNP's P-value from the same file (columns 1 and 8).

In [ ]:
pd.DataFrame({"label": ["p5e-8", "p1e-5", "p1e-3", "p0.05", "p1"], "lower": 0,
              "upper": [5e-8, 1e-5, 1e-3, 0.05, 1]}).to_csv("ranges.txt", sep=" ", header=False, index=False)

In [ ]:
%%bash
plink --bfile target --score base_clumped.txt 1 4 6 header sum \
      --q-score-range ranges.txt base_clumped.txt 1 8 header --out ct > /dev/null
head -3 ct.p1e-3.profile

**Step 3: choose the threshold in the training half.** For each threshold we compute the AUC in the training
individuals only.

In [ ]:
def auc(score, y):
    r = rankdata(score); n1 = y.sum(); n0 = len(y) - n1
    return (r[y == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)

train = set(pd.read_csv("train.txt", sep=" ", header=None)[1])
test = set(pd.read_csv("test.txt", sep=" ", header=None)[1])
labels = ["p5e-8", "p1e-5", "p1e-3", "p0.05", "p1"]

rows = []
for lab, upper in zip(labels, [5e-8, 1e-5, 1e-3, 0.05, 1]):
    prof = pd.read_csv(f"ct.{lab}.profile", sep=r"\s+")
    tr = prof[prof.IID.isin(train)]
    rows.append((f"P < {upper:g}", (clumped.P <= upper).sum(), round(auc(tr.SCORESUM, (tr.PHENO == 2).astype(int)), 3)))
pd.DataFrame(rows, columns=["threshold", "SNPs", "AUC (training)"])

**Exercise 5.** Which threshold gives the highest AUC in the training half? (The differences are small; that is typical.) Now compute the AUC of **that one
score** in the test half (`prof[prof.IID.isin(test)]`). Then compute the test AUC for all five thresholds.

- Is the best threshold in the test half the same one?
- Why would choosing the threshold *and* reporting its AUC in the same individuals overstate how well the score
  works? (Think about the three-sample design in *Calculating PRS: workflow and data*.)

In [ ]:
# Exercise 5

**Exercise 6 (optional).** Repeat Step 2 **without** clumping: use `base_qc.txt` in place of
`base_clumped.txt` (both times) and `--out noclump`. How many SNPs pass $P < 0.05$ now, and how does the test AUC compare with the clumped score at the
same threshold? Explain the difference using what you know about linkage disequilibrium.

In [ ]:
# Exercise 6